### Import Dependencies

In [ ]:
import openai
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree


### Load Environment
from dotenv import load_dotenv #import 
import os 

load_dotenv("../../.env")



### Define embedding function

In [ ]:
@traceable(
    name="embed_query",
    run_type="embedding", # tells langsmith we are actually running embedding text -> specific set of vectors
    metadata={
        "ls_provider": "openai", 
              "model": "text-embedding-3-small"
    } ## langsmith needs this to calculate cost of runs
)
def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    
    current_run  = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens,
        }
    return response.data[0].embedding

### Retrieval Function

In [ ]:
qdrant_client = QdrantClient(
    url="http://localhost:6333"
)

In [ ]:
@traceable(
    name="retrieve_data",
    run_type="retriever"
)
def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=embedding,
        limit=k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

### Format retrieved context

In [ ]:
@traceable(
    name="format_retrieved_context",
    run_type="prompt"
)
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context
 

### Create prompt template function

In [ ]:
@traceable(
    name="build_prompt",
    run_type="prompt"
)
def build_prompt(preprocessed_context, question): # question is the users question...
    prompt = f"""You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt


### Generate answer

In [ ]:
@traceable(
    name="generate_answer",
    run_type="llm", # we are running an llm call
    metadata={"ls_provider": "openai", "model": "gpt-5-nano"} ## langsmith needs this to calculate cost of runs
)
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="minimal"
    )
    current_run  = get_current_run_tree()
    if current_run: # of tokens used for input and how many used for output
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        }
    return response.choices[0].message.content

In [ ]:
@traceable(
    name="rag_pipeline"
)
def rag_pipeline(question, topk_k=5):
    data_for_question = retrieve_data(question, k=topk_k)
    processed_context = process_context(data_for_question)
    prompt = build_prompt(processed_context, question)
    return generate_answer(prompt)

print(rag_pipeline("Do you have a USB connectable fan for hot summers."))